## Simulate Mistral 3B as participant in code switching

### Import and process the items

In [1]:
import pandas as pd
import os, openpyxl, asyncio, anthropic
from pathlib import Path
from openai import OpenAI
from typing import Optional
import concurrent.futures


BASE_DIR = Path.cwd().parent
INPUT  = BASE_DIR / "Human_data" / "coding_clean_final.xlsx"
OUTPUT = BASE_DIR / "LLM_data" 

df_human = pd.read_excel(INPUT)

def build_participant_items(df: pd.DataFrame) -> dict:
    participant_items = {}
    for pid, group in df.groupby("ID", sort=False):
        participant_items[pid] = group[["Item", "Itemtype"]] \
                                      .to_dict(orient="records")
    return participant_items

participant_items = build_participant_items(df_human)

### Request LLM

In [2]:
# initial setup
MODEL_NAME = "anthropic/claude-opus-4.6"  
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
    timeout=60,
    max_retries=3
 )

# system prompt
SYSTEM_PROMPT = """You are a Spanish-English bilingual. You are participating in a linguistic task.

In this study, you will read sentences in English, Spanish or a mixture of Spanish and English. This study is completing sentences.

You will see sentences that are missing the last word. You are asked to complete each sentence in a plausible way with the first word that comes to mind. You are asked to complete only one word.

There are three sentence types. The following is an example of an English sentence. Here you would need to complete an English word.

Martha wants to buy groceries and goes to the ...

A possible completion could be the following:
Martha wants to buy groceries and goes to the ...
supermarket.

Una oración también puede estar completamente en español. A continuación, se muestra un ejemplo. Aquí, se escribe una palabra en español para completar la oración:
Marta quiere hacer compras y va al ...
supermercado.

Una oración también puede empezar in Spanish and change to English, as in the following example. Here you would complete an English word:
Marta quiere hacer compras and goes to the ...
supermarket.

Please write down what first comes to mind. Please complete the sentence with only one word. Please complete a Spanish sentence with a Spanish word, and the English and mixed sentences with an English word. Your completion needs to be plausible and grammatically correct, but other than that, you can write anything.

Important: only return the word."""

In [3]:
# define functions to run by participant
def run_single_trial(participant_id: str, item: dict) -> dict:
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            max_tokens=10,
            temperature=0.1,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": item["Item"]}
            ]
        )
        return {
            "ID":         participant_id,
            "Item":       item["Item"],
            "Itemtype":   item["Itemtype"],
            "Completion": response.choices[0].message.content.strip(),
            "status":     "success"
        }
    except Exception as e:
        return {
            "ID":         participant_id,
            "Item":       item["Item"],
            "Itemtype":   item["Itemtype"],
            "Completion": None,
            "status":     str(e)
        }

def run_participant(participant_id: str, items: list[dict],
                    max_workers: int = 5) -> list[dict]:
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(run_single_trial, participant_id, item)
            for item in items
        ]
        return [f.result() for f in concurrent.futures.as_completed(futures)]

def run_all(participant_items: dict, max_workers: int = 5) -> pd.DataFrame:
    all_results = []
    participants = list(participant_items.items())
    total = len(participants)
    
    for i, (pid, items) in enumerate(participants, 1):
        print(f"Running participant {i}/{total}: {pid}")
        results = run_participant(pid, items, max_workers)
        all_results.extend(results)
        
        # checkpoint every 10 participants
        if i % 10 == 0:
            df_checkpoint = pd.DataFrame(all_results)
            checkpoint_path = OUTPUT / f"checkpoint_{i}of{total}.csv"
            df_checkpoint.to_csv(checkpoint_path, index=False)  
            print(f"  ✓ Checkpoint: {checkpoint_path.name}")
    
    return pd.DataFrame(all_results)  

In [5]:
# Run all participants and get the results in a DataFrame
df_llm = run_all(participant_items)

Running participant 1/120: 1750873464
Running participant 2/120: 1750874407
Running participant 3/120: 1750875716
Running participant 4/120: 1750875995
Running participant 5/120: 1750879780
Running participant 6/120: 1750887668
Running participant 7/120: 1751968832
Running participant 8/120: 1751969612
Running participant 9/120: 1751969748
Running participant 10/120: 1751970033
  ✓ Checkpoint: checkpoint_10of120.csv
Running participant 11/120: 1751971669
Running participant 12/120: 1751973483
Running participant 13/120: 1751974229
Running participant 14/120: 1751977115
Running participant 15/120: 1751978691
Running participant 16/120: 1751978981
Running participant 17/120: 1751980278
Running participant 18/120: 1751980488
Running participant 19/120: 1751981228
Running participant 20/120: 1751981332
  ✓ Checkpoint: checkpoint_20of120.csv
Running participant 21/120: 1751981512
Running participant 22/120: 1751983542
Running participant 23/120: 1751986873
Running participant 24/120: 175198

### Check and save the output data

In [6]:
# check failed trial
failed = df_llm[df_llm["status"] != "success"]
if len(failed) > 0:
    print(f"⚠️ failed trial: {len(failed)}个")
    print(failed[["ID", "Item", "status"]])
else:
    print(f"✓ all success，in total of {len(df_llm)} trials")
# delete status column and save to excel
df_llm = df_llm.drop(columns=["status"])
df_llm.to_csv(OUTPUT / f"llm_{MODEL_NAME.replace('/', '-')}.csv", index=False)

⚠️ failed trial: 26个
              ID                                      Item  \
234   1750879780  Der biologische Dünger helped enrich the   
519   1751971669  Der biologische Dünger helped enrich the   
795   1751980278  Der biologische Dünger helped enrich the   
861   1751980488  Der biologische Dünger helped enrich the   
1011  1751983542  Der biologische Dünger helped enrich the   
1085  1751986873  Der biologische Dünger helped enrich the   
1126  1751987893  Der biologische Dünger helped enrich the   
1226  1751988192  Der biologische Dünger helped enrich the   
1290  1751988198  Der biologische Dünger helped enrich the   
1370  1751991220  Der biologische Dünger helped enrich the   
1633  1752003672  Der biologische Dünger helped enrich the   
1692  1752004990  Der biologische Dünger helped enrich the   
1937  1752051043  Der biologische Dünger helped enrich the   
1969  1752051600  Der biologische Dünger helped enrich the   
2250  1752058440  Der biologische Dünger helped e

### test

In [4]:
# ── use the first participant for testing ─────────────────────────────────────
first_pid = list(participant_items.keys())[0]
first_items = participant_items[first_pid]

for item in first_items[:3]:
    print(f"  {item}")

# ── only run this participant ───────────────────────────────────────
#test single participant
test_results = run_participant(first_pid, first_items)
df_test = pd.DataFrame(test_results)
print(df_test[["Itemtype", "Completion", "status"]])

  {'Item': 'Everyone was honking at the old man who was driving too', 'Itemtype': 'English'}
  {'Item': 'Die junge Mutter aus der Nachbarschaft fütterte her baby some warm', 'Itemtype': 'code_switching'}
  {'Item': 'The man downstairs played his stereo much too', 'Itemtype': 'English'}
          Itemtype Completion   status
0          English       hour  success
1          English      loud.  success
2          English      slow.  success
3   code_switching       milk  success
4   code_switching      names  success
5          English        cry  success
6          English      raise  success
7          English   families  success
8   code_switching        key  success
9   code_switching    stripes  success
10  code_switching     accent  success
11         English       sink  success
12         English       nose  success
13         English       door  success
14  code_switching       barn  success
15  code_switching       name  success
16  code_switching      paper  success
17         